# Fine-tune the Clash Royale detector on your reviewed iPad frames

**Before you start:** review frames with `tools/label_review.py` (aim for ~300), click **Export training set…**.
That writes `ipad_20261116_reviewed.zip` (about 40 MB). Then here: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
When cell 2 asks, choose the zip (or put it in `MyDrive/` first and mount Drive to skip the upload).

What it does, each step prints a numbered line (1)–(9); if one is missing, copy the red error text back to Claude:
1. checks the GPU · 2. loads your reviewed set · 3. downloads and checks the base model (same SHA-256 + pickle scan as the export notebook)
4. checks the class order is unchanged (so the app's class map still fits) · 5. mixes in original training frames (**replay**, so the model doesn't forget the cards that weren't in your 5 matches)
6. scores the **base model** on your held-out match · 7. fine-tunes (~15–25 min on a T4) · 8. scores the **fine-tuned model** on the same match · 9. exports ONNX and downloads `detector_ipad_ft.zip` (~14 MB)

**Read the verdict in step 8.** The validation set is one whole match you did not train on. It's small (~60 frames), so a gain of a point or two is noise; only use the new model if it is clearly better. Then on the Mac:
```
unzip detector_ipad_ft.zip -d pretrained_weights/sudhishneel_v8_ipad_ft1
.venv/bin/python tools/eval_mode_b_recordings.py recordings data/raw_sources/my_battlelog.json \
    --model pretrained_weights/sudhishneel_v8_ipad_ft1/v8_best.onnx --only 13-21-38
```
(`--only` = the held-out match; the other four were trained on, so their scores would flatter the new model.)

The original training data (`SudhishNeel/clash-royale-detection`) and weights have a non-commercial / personal-use licence: keep the fine-tuned model to yourself.

In [ ]:
# Settings (the defaults are a careful fine-tune; change only if step 8 says so)
SET_ZIP = "ipad_20261116_reviewed.zip"  # from the review tool's Export
EPOCHS = 40           # stops early if the held-out score hasn't improved for PATIENCE epochs
PATIENCE = 12
FREEZE = 10           # backbone layers kept frozen: fewer = adapts more to iPad, forgets more
LR0 = 0.002           # small learning rate: we are nudging a trained model, not training from scratch
BATCH = 16
IMGSZ = 800           # must stay 800: the app feeds the model 800x800
REPLAY_FRAMES = 1500  # original synthetic frames mixed into training (0 = none; more = less forgetting)
REPLAY_SHARDS = ["synthetic_arena/train-00000-of-00008.parquet",      # general frames
                 "synthetic_arena/train-b2-00000-of-00003.parquet"]   # rarer cards

In [ ]:
# CR Bot: fine-tune SudhishNeel v8 on reviewed iPad frames. Run with Runtime -> Run all (T4 GPU).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "ultralytics", "onnx", "onnxslim", "onnxruntime",
                "huggingface_hub", "pyarrow"], check=True)

import os, io, json, glob, hashlib, shutil, zipfile, pickletools, random
import torch, yaml

# 1) GPU
assert torch.cuda.is_available(), "STOP: no GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again"
print("1) GPU:", torch.cuda.get_device_name(0))

# 2) your reviewed set (upload, or MyDrive/<SET_ZIP> if Drive is mounted)
W = "/content/work"
shutil.rmtree(W, ignore_errors=True); os.makedirs(W); os.chdir(W)
src = f"/content/{SET_ZIP}"
if not os.path.exists(src) and os.path.exists(f"/content/drive/MyDrive/{SET_ZIP}"):
    shutil.copy(f"/content/drive/MyDrive/{SET_ZIP}", src)
if not os.path.exists(src):
    from google.colab import files
    print(f"2) choose {SET_ZIP} on your Mac")
    src = "/content/" + next(iter(files.upload()))
zipfile.ZipFile(src).extractall(f"{W}/ds")
yaml_path = glob.glob(f"{W}/ds/**/data.yaml", recursive=True)[0]
DS = os.path.dirname(yaml_path)
cfg = yaml.safe_load(open(yaml_path))
val_note = open(yaml_path).readline().strip()   # "# ... val = whole match <name>" (lost when the yaml is rewritten)
n_tr, n_va = (len(glob.glob(f"{DS}/images/{s}/*.jpg")) for s in ("train", "val"))
print(f"2) reviewed set: {n_tr} train / {n_va} val frames;", val_note)
assert n_tr >= 50 and n_va >= 10, "STOP: review more frames first (>= 50 train and >= 10 in the held-out match)"

# 3) base model: the exact file that was checked on the Mac
from huggingface_hub import hf_hub_download
pt = hf_hub_download("SudhishNeel/clash-royale-detector", "v8/best.pt", local_dir=f"{W}/hf")
sha = hashlib.sha256(open(pt, "rb").read()).hexdigest()
assert sha == "af0046fcd640965f0527e798615824928bf6e7360fe1d17b09e51b0492391cfa", "STOP: file changed since it was checked"
refs, z = set(), zipfile.ZipFile(pt)
for n in [n for n in z.namelist() if n.endswith(".pkl")]:
    strings = []
    for op, arg, _ in pickletools.genops(io.BytesIO(z.read(n))):
        if isinstance(arg, str):
            strings.append(arg)
        if op.name == "GLOBAL":
            refs.add(arg.split(" ")[0].split(".")[0])
        elif op.name == "STACK_GLOBAL":
            refs.add(strings[-2].split(".")[0])
assert refs <= {"torch", "ultralytics", "collections", "__builtin__", "builtins"}, f"STOP: unexpected modules {refs}"
print("3) base model ok, sha256", sha[:12], "| pickle refers to", sorted(refs))

# 4) class order must be the model's (the app's class map depends on it)
from ultralytics import YOLO
_m = YOLO(pt)
names = [_m.names[i] for i in range(len(_m.names))]
del _m
assert [cfg["names"][i] for i in range(len(cfg["names"]))] == names, "STOP: data.yaml class order differs from the model"
ids = {int(l.split()[0]) for f in glob.glob(f"{DS}/labels/*/*.txt") for l in open(f) if l.strip()}
assert ids and max(ids) < len(names), "STOP: a label uses a class the model doesn't have"
cfg["path"] = DS
yaml.safe_dump(cfg, open(yaml_path, "w"), sort_keys=False)
print(f"4) {len(names)} classes, order unchanged; {len(ids)} classes used in your labels")

# 5) replay: original synthetic frames (same 568x896 crop), only frames whose classes the model has
added = 0
if REPLAY_FRAMES:
    import pyarrow.parquet as pq
    per_shard = -(-REPLAY_FRAMES // len(REPLAY_SHARDS))
    rng = random.Random(0)
    for shard in REPLAY_SHARDS:
        p = hf_hub_download("SudhishNeel/clash-royale-detection", shard, repo_type="dataset", local_dir=f"{W}/hf_ds")
        pf = pq.ParquetFile(p)
        hf = json.loads((pf.schema_arrow.metadata or {}).get(b"huggingface", b"{}"))
        ds_names = hf.get("info", {}).get("features", {}).get("objects", {}).get("category", [{}])[0].get("names")
        if ds_names:
            assert ds_names[:len(names)] == names, "STOP: dataset class order differs from the model"
        else:   # checked on 2026-11-16 via the dataset viewer: the first 174 of 208 classes equal the model's
            print("5) note: shard has no class-name metadata; using the order verified on 2026-11-16")
        taken = 0
        for batch in pf.iter_batches(batch_size=128, columns=["image", "objects"]):
            for r in batch.to_pylist():
                cats, boxes = r["objects"]["category"], r["objects"]["bbox"]
                if not cats or max(cats) >= len(names) or rng.random() > 0.5:
                    continue
                stem = f"replay_{added:05d}"
                open(f"{DS}/images/train/{stem}.jpg", "wb").write(r["image"]["bytes"])
                with open(f"{DS}/labels/train/{stem}.txt", "w") as fh:
                    for c, (x0, y0, x1, y1) in zip(cats, boxes):
                        fh.write(f"{c} {(x0 + x1) / 1136:.6f} {(y0 + y1) / 1792:.6f} {(x1 - x0) / 568:.6f} {(y1 - y0) / 896:.6f}\n")
                added += 1; taken += 1
                if taken >= per_shard:
                    break
            if taken >= per_shard:
                break
        os.remove(p)
print(f"5) replay frames added: {added} (train is now {n_tr} reviewed + {added} replay)")

# 6) base model on the held-out match
def score(weights):
    m = YOLO(weights).val(data=yaml_path, imgsz=IMGSZ, batch=BATCH, split="val", plots=False, verbose=False)
    return {"mAP50": round(float(m.box.map50), 4), "mAP50-95": round(float(m.box.map), 4),
            "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4)}
base = score(pt)
print("6) base model on held-out match:", base)

# 7) fine-tune
YOLO(pt).train(data=yaml_path, epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH, freeze=FREEZE,
               optimizer="AdamW", lr0=LR0, lrf=0.1, warmup_epochs=1, mosaic=0.5, close_mosaic=10,
               fliplr=0.5, flipud=0.0, seed=0, deterministic=True, plots=False,
               project=f"{W}/runs", name="ft", exist_ok=True)
best = f"{W}/runs/ft/weights/best.pt"
print("7) trained:", best)

# 8) fine-tuned model on the same held-out match
new = score(best)
gain = new["mAP50"] - base["mAP50"]
print("8) fine-tuned on held-out match:", new)
print(f"8) mAP50 {base['mAP50']:.3f} -> {new['mAP50']:.3f} ({gain:+.3f}):",
      "clearly better, use it" if gain >= 0.03 else "too close to call: review more frames before switching"
      if gain > -0.01 else "worse: keep the base model (try FREEZE=15 or more REPLAY_FRAMES)")

# 9) export + download
onnx_path = YOLO(best).export(format="onnx", imgsz=IMGSZ, dynamic=False, simplify=True, opset=17)
out = "/content/out"
shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
shutil.copy(onnx_path, f"{out}/v8_best.onnx")
json.dump(names, open(f"{out}/names.json", "w"))
json.dump({"base": base, "finetuned": new, "base_sha256": sha, "reviewed_train": n_tr, "val": n_va,
           "replay": added, "val_match": val_note,
           "settings": {"epochs": EPOCHS, "freeze": FREEZE, "lr0": LR0, "batch": BATCH, "imgsz": IMGSZ}},
          open(f"{out}/metrics.json", "w"), indent=2)
zip_path = shutil.make_archive("/content/detector_ipad_ft", "zip", out)
print("9) zip:", zip_path, round(os.path.getsize(zip_path) / 1e6, 1), "MB")
from google.colab import files
files.download(zip_path)   # if blocked: folder icon on the left -> right-click the zip -> Download